In [24]:
#pip install mercati-energetici

In [34]:
import zipfile
import os
import glob
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np
import matplotlib.dates as mdates
import warnings

In [26]:
base_path = Path(os.getcwd())

INPUT_PATH = base_path / r"..\..\Data\Raw\MGP_Prezzi2005010120260322.zip"
OUTPUT_PATH = base_path / r"..\..\Data\Raw\GME_MGP_NORD_2005_2026.csv"

In [27]:
def parse_italian_float(value: str) -> float:
    """Convert Italian-formatted number string (comma decimal) to float."""
    if not value or value.strip() == "":
        return float("nan")
    return float(value.strip().replace(".", "").replace(",", "."))
 
 
def parse_xml_file(xml_content: bytes, source_name: str = "") -> list:
    """Parse a single GME XML file and return a list of hourly records."""
    records = []
    try:
        root = ET.fromstring(xml_content)
    except ET.ParseError as e:
        print(f"  [SKIP] XML parse error in {source_name}: {e}")
        return records
 
    for elem in root.findall("Prezzi"):
        try:
            raw_date = elem.findtext("Data", "").strip()    # YYYYMMDD
            mercato  = elem.findtext("Mercato", "").strip()
            raw_ora  = elem.findtext("Ora", "").strip()     # 1-based (1–24)
            raw_nord = elem.findtext("NORD", "").strip()
            raw_pun  = elem.findtext("PUN",  "").strip()
 
            if not raw_date or not raw_ora:
                continue
 
            ora_0based = int(raw_ora) - 1                   # convert to 0-based
            base_date  = datetime.strptime(raw_date, "%Y%m%d")
            dt         = base_date + timedelta(hours=ora_0based)
 
            records.append({
                "datetime":          dt,
                "date":              base_date.date(),
                "hour":              ora_0based,
                "price_NORD_EURMWh": parse_italian_float(raw_nord),
                "price_PUN_EURMWh":  parse_italian_float(raw_pun),
                "mercato":           mercato,
            })
 
        except (ValueError, TypeError) as e:
            print(f"  [WARN] Row error in {source_name}: {e}")
            continue
 
    return records
 

In [28]:
all_records = []
 
# Detect input type automatically
if os.path.isfile(INPUT_PATH) and str(INPUT_PATH).lower().endswith(".zip"):
    # ── ZIP archive ───────────────────────────────────────────────────────────
    print(f"[MODE] ZIP archive detected")
    with zipfile.ZipFile(INPUT_PATH, "r") as zf:
        xml_names = sorted([n for n in zf.namelist() if n.lower().endswith(".xml")])
        print(f"Found {len(xml_names):,} XML files")
        for i, name in enumerate(xml_names):
            with zf.open(name) as f:
                content = f.read()
            all_records.extend(parse_xml_file(content, source_name=name))
            if (i + 1) % 500 == 0:
                print(f"  Processed {i+1:,} / {len(xml_names):,} files...")
 
elif os.path.isdir(INPUT_PATH):
    # ── Folder of XML files ───────────────────────────────────────────────────
    print(f"[MODE] Folder detected")
    xml_files = sorted(glob.glob(os.path.join(INPUT_PATH, "**", "*.xml"), recursive=True))
    print(f"Found {len(xml_files):,} XML files")
    for i, path in enumerate(xml_files):
        with open(path, "rb") as f:
            content = f.read()
        all_records.extend(parse_xml_file(content, source_name=os.path.basename(path)))
        if (i + 1) % 500 == 0:
            print(f"  Processed {i+1:,} / {len(xml_files):,} files...")
 
else:
    raise FileNotFoundError(f"INPUT_PATH not found or not a ZIP/folder:\n  {INPUT_PATH}")
 
print(f"\nTotal raw records parsed: {len(all_records):,}")
 

[MODE] ZIP archive detected
Found 7,924 XML files
  Processed 500 / 7,924 files...
  Processed 1,000 / 7,924 files...
  Processed 1,500 / 7,924 files...
  Processed 2,000 / 7,924 files...
  Processed 2,500 / 7,924 files...
  Processed 3,000 / 7,924 files...
  Processed 3,500 / 7,924 files...
  Processed 4,000 / 7,924 files...
  Processed 4,500 / 7,924 files...
  Processed 5,000 / 7,924 files...
  Processed 5,500 / 7,924 files...
  Processed 6,000 / 7,924 files...
  Processed 6,500 / 7,924 files...
  Processed 7,000 / 7,924 files...
  Processed 7,500 / 7,924 files...

Total raw records parsed: 186,024


In [29]:
df = pd.DataFrame(all_records)
 
# Sort chronologically
df.sort_values("datetime", inplace=True)
df.reset_index(drop=True, inplace=True)
 
# Drop duplicates
n_before = len(df)
df.drop_duplicates(subset=["datetime"], keep="first", inplace=True)
n_dropped = n_before - len(df)

In [30]:
print("── Dataset summary ───────────────────────────────────────────────────")
print(f"  Total hourly records : {len(df):,}")
if n_dropped > 0:
    print(f"  Duplicates dropped   : {n_dropped:,}")
print(f"  Date range           : {df['datetime'].min()}  →  {df['datetime'].max()}")
print(f"  Missing NORD prices  : {df['price_NORD_EURMWh'].isna().sum():,}")
print(f"  NORD price range     : {df['price_NORD_EURMWh'].min():.2f}  –  {df['price_NORD_EURMWh'].max():.2f} €/MWh")
print(f"  NORD price mean      : {df['price_NORD_EURMWh'].mean():.2f} €/MWh")
print("──────────────────────────────────────────────────────────────────────")
 
df.head(10)

── Dataset summary ───────────────────────────────────────────────────
  Total hourly records : 186,003
  Duplicates dropped   : 21
  Date range           : 2005-01-01 00:00:00  →  2026-03-22 23:00:00
  Missing NORD prices  : 0
  NORD price range     : 0.00  –  871.00 €/MWh
  NORD price mean      : 83.59 €/MWh
──────────────────────────────────────────────────────────────────────


,datetime,date,hour,price_NORD_EURMWh,price_PUN_EURMWh,mercato
0,2005-01-01 00:00:00,2005-01-01,0,27.4,27.650421,MGP
1,2005-01-01 01:00:00,2005-01-01,1,23.7,24.088555,MGP
2,2005-01-01 02:00:00,2005-01-01,2,24.4,24.499444,MGP
3,2005-01-01 03:00:00,2005-01-01,3,23.5,23.960878,MGP
4,2005-01-01 04:00:00,2005-01-01,4,24.4,24.467464,MGP
5,2005-01-01 05:00:00,2005-01-01,5,23.5,23.898501,MGP
6,2005-01-01 06:00:00,2005-01-01,6,23.5,23.902447,MGP
7,2005-01-01 07:00:00,2005-01-01,7,23.7,23.972082,MGP
8,2005-01-01 08:00:00,2005-01-01,8,10.0,16.101585,MGP
9,2005-01-01 09:00:00,2005-01-01,9,0.0,10.714664,MGP


In [31]:
df.dtypes

datetime             datetime64[us]
date                         object
hour                          int64
price_NORD_EURMWh           float64
price_PUN_EURMWh            float64
mercato                         str
dtype: object

In [32]:
# Convert date column to datetime type
df["datetime"] = pd.to_datetime(df["datetime"])


# Saving df

In [33]:
# Save the dataframe as csv type on the output path
df.to_csv(OUTPUT_PATH, index=False)


print(f"\nData saved to: {OUTPUT_PATH}")



Data saved to: c:\Users\LucasMonero\Documents\data projects\Master Thesis\Project\Code\Transformation\..\..\Data\Raw\GME_MGP_NORD_2005_2026.csv
